# Task 2

---

## Predictive modeling of customer bookings

This Jupyter notebook includes some code to get you started with this predictive modeling task. We will use various packages for data manipulation, feature engineering and machine learning.

### Exploratory data analysis

First, we must explore the data in order to better understand what we have and the statistical properties of the dataset.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("customer_booking.csv", encoding="ISO-8859-1")
df.head()

The `.head()` method allows us to view the first 5 rows in the dataset, this is useful for visual inspection of our columns

In [ ]:
df.info()

The `.info()` method gives us a data description, telling us the names of the columns, their data types and how many null values we have. Fortunately, we have no null values. It looks like some of these columns should be converted into different data types, e.g. flight_day.

To provide more context, below is a more detailed data description, explaining exactly what each column means:

- `num_passengers` = number of passengers travelling
- `sales_channel` = sales channel booking was made on
- `trip_type` = trip Type (Round Trip, One Way, Circle Trip)
- `purchase_lead` = number of days between travel date and booking date
- `length_of_stay` = number of days spent at destination
- `flight_hour` = hour of flight departure
- `flight_day` = day of week of flight departure
- `route` = origin -> destination flight route
- `booking_origin` = country from where booking was made
- `wants_extra_baggage` = if the customer wanted extra baggage in the booking
- `wants_preferred_seat` = if the customer wanted a preferred seat in the booking
- `wants_in_flight_meals` = if the customer wanted in-flight meals in the booking
- `flight_duration` = total duration of flight (in hours)
- `booking_complete` = flag indicating if the customer completed the booking

Before we compute any statistics on the data, lets do any necessary data conversion

In [ ]:
df["flight_day"].unique()

In [ ]:
mapping = {
    "Mon": 1,
    "Tue": 2,
    "Wed": 3,
    "Thu": 4,
    "Fri": 5,
    "Sat": 6,
    "Sun": 7,
}

df["flight_day"] = df["flight_day"].map(mapping)

In [ ]:
df["flight_day"].unique()

In [ ]:
df.describe()

The `.describe()` method gives us a summary of descriptive statistics over the entire dataset (only works for numeric columns). This gives us a quick overview of a few things such as the mean, min, max and overall distribution of each column.

From this point, you should continue exploring the dataset with some visualisations and other metrics that you think may be useful. Then, you should prepare your dataset for predictive modelling. Finally, you should train your machine learning model, evaluate it with performance metrics and output visualisations for the contributing variables. All of this analysis should be summarised in your single slide.

In [ ]:
for column in ["sales_channel", "trip_type", "flight_day"]:
    print(f"\nBooking rate by {column}:")

    print((df.groupby(column)["booking_complete"].mean() * 100).round(1))


These numbers are booking percentages for each group. For example, 15.5% of customers using the Internet channel booked.

Sales channel: Internet customers booked more often than Mobile customers: 15.5% vs 10.8%.
Trip type: Round-trip customers booked more often: 15.1%, compared with 5.2% for OneWay and 4.3% for CircleTrip.
Flight day: Booking rates are quite similar across the week, ranging from 14.1% to 16.3%. Day 3 is Wednesday if your notebook maps Monday to 1.
These are patterns in this dataset. They don’t prove that the channel or trip type caused someone to book, but they may help the model make predictions.

In [ ]:
# Find the 10 countries with the most bookings
top_origins = df["booking_origin"].value_counts().head(10).index

# Keep only customers from those 10 countries
top_customers = df[df["booking_origin"].isin(top_origins)]

# Calculate the booking percentage for each country
origin_rates = (
    top_customers.groupby("booking_origin")["booking_complete"].mean() * 100
)

# Show the percentages, highest first, rounded to one decimal place
print(origin_rates.round(1).sort_values(ascending=False))

These are the **booking percentages for customers from each country**. For example, of the customers whose booking origin was Malaysia, **34.4% completed a booking**.

Malaysia had the highest rate in this list (**34.4%**), while Australia and New Zealand had the lowest (**5.0%** each). This suggests a customer’s booking origin may help the model, but it doesn’t prove that a person’s country caused them to book.


In [ ]:
# Group the number of days before the flight into ranges
df["purchase_lead_group"] = pd.cut(
    df["purchase_lead"],
    bins=[-1, 7, 30, 90, 180, float("inf")],
    labels=["0–7 days", "8–30 days", "31–90 days", "91–180 days", "181+ days"]
)

# Calculate the booking percentage for each range
lead_rates = df.groupby("purchase_lead_group", observed=False)["booking_complete"].mean() * 100

# Display the percentages, rounded to one decimal place
print(lead_rates.round(1))


This shows the percentage of customers who booked in each time range:

Customers booking 0–7 days before the flight had the highest booking rate: 17.3%.
The rate generally gets lower for customers booking further ahead, reaching 13.4% for 91–180 days.
For customers booking more than 181 days ahead, it rises slightly to 14.2%.
So, in this dataset, customers booking close to their flight were a little more likely to complete a booking. These are patterns in the data; they don’t prove that booking time causes someone to book.

In [ ]:
# Group the length of stay into ranges of nights
df["stay_group"] = pd.cut(
    df["length_of_stay"],
    bins=[-1, 3, 7, 14, 30, float("inf")],
    labels=["0–3 nights", "4–7 nights", "8–14 nights", "15–30 nights", "31+ nights"]
)

# Calculate the booking percentage for each range
stay_rates = df.groupby("stay_group", observed=False)["booking_complete"].mean() * 100

# Display the results, rounded to one decimal place
print(stay_rates.round(1))

he percentages show the share of customers in each stay group who completed a booking:
- 4–7 nights: 19.4% — the highest rate
- 0–3 nights: 17.8%
- 15–30 nights: 10.1%
- 31+ nights: 11.9%
NaN for 8–14 nights isn’t an error. Your dataset has no customers with stays in that range, so there’s no booking rate to calculate for it.

In [ ]:
# Remove the temporary groups used for exploring the data
df_model = df.drop(
    columns=["purchase_lead_group", "stay_group"],
    errors="ignore"
)

# X contains the information the model will use
X = df_model.drop(columns=["booking_complete"])

# y contains the answer the model will learn to predict
y = df_model["booking_complete"]

# Check the size of the inputs and answers
print("Input data:", X.shape)
print("Target data:", y.shape)

This output means the split worked:
- Input data (50000, 13): 50,000 customer records, each with 13 details for the model to learn from.
- Target data (50000,): 50,000 booking results, one matching each customer record. 1 means booked; 0 means not booked.
Next, we’ll prepare the text answers in X—such as Internet and RoundTrip—so the model can use them.

In [ ]:
# let’s identify which columns contain words. The model needs these converted into numbers late

# Find columns that contain text, such as "Internet" or "RoundTrip"
text_columns = X.select_dtypes(include="object").columns.tolist()

# Show their names
print("Text columns:", text_columns)


Those are the four text columns the model can’t use as words. Later, we’ll convert their categories into numbers. For now, split the data into one part to train the model and one part to test it.

In [ ]:
from sklearn.model_selection import train_test_split

# Keep 20% of the data aside for testing
# stratify=y keeps the booking percentages similar in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))


The split worked:

40,000 rows are set aside for the model to learn from.
10,000 rows are kept aside to check how well it learned.
Next, convert the text categories into numbers. This code learns the conversion from the training data, then applies it to both sets:

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Convert text categories into numeric columns.
# Keep all other columns as they are.
preprocessor = ColumnTransformer(
    transformers=[
        ("text", OneHotEncoder(handle_unknown="ignore"), text_columns)
    ],
    remainder="passthrough"
)

# Learn the conversions from the training data
X_train_ready = preprocessor.fit_transform(X_train)

# Apply the same conversions to the testing data
X_test_ready = preprocessor.transform(X_test)

print("Training data shape:", X_train_ready.shape)
print("Testing data shape:", X_test_ready.shape)

That output is expected. The text columns have been expanded into number columns, so the data now has 910 columns instead of 13. Both the training and testing data have the same 910 columns, so the model can use them consistently.

The data is ready for a first model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create a model; balanced weights help account for fewer bookings than non-bookings
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

# Let the model learn from the training data
model.fit(X_train_ready, y_train)

That output means the Random Forest model was created. If it appeared after model.fit(X_train_ready, y_train), it also means training completed.
Now ask it to predict bookings for the 10,000 test examples and compare its predictions with the real answers:

In [ ]:
# Ask the model to predict the test results
y_pred = model.predict(X_test_ready)

# Compare its predictions with the correct answers
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

The model is good at spotting customers who won’t book, but it misses most customers who do book.
- Booking (1) recall = 0.12: it found only about 12 out of every 100 actual bookings—around 180 of the 1,496 bookings in the test data.
- Booking (1) precision = 0.49: when it predicted “this customer will book,” it was correct about half the time.
- Accuracy = 0.85: it got 85% of all predictions right, but this looks better than the model really is because most customers in the data didn’t book.
So, the model needs improvement at finding likely bookers. The next useful check is a confusion matrix, which shows how many bookings it found and missed.

In [ ]:
from sklearn.metrics import confusion_matrix

# Compare the real answers with the model's predictions
cm = confusion_matrix(y_test, y_pred)

print(cm)


Here’s what the matrix shows:

8,324 customers who didn’t book were correctly identified.
180 customers were predicted to book, but didn’t.
1,320 customers actually booked, but the model missed them.
176 customers who booked were correctly identified.
The key issue is that the model found only 176 of the 1,496 actual bookings. So, despite its 85% accuracy, it currently misses most potential bookers.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# Convert text columns into numbers; keep the other columns as they are
preprocessor = ColumnTransformer(
    transformers=[
        ("text", OneHotEncoder(handle_unknown="ignore"), text_columns)
    ],
    remainder="passthrough"
)

# Put the text conversion and Random Forest together
cv_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Make five rounds, keeping the booking and non-booking proportions similar
five_folds = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Measure performance in each round, using the training data
scores = cross_validate(
    cv_model,
    X_train,
    y_train,
    cv=five_folds,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
)

# Show the average score across the five rounds
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    values = scores[f"test_{metric}"]
    print(f"{metric}: {values.mean() * 100:.1f}% average")


These are the model’s average scores across five tests:
- Accuracy: 85.2% — it got this share of all predictions right. Because most customers didn’t book, accuracy alone can be misleading.
- Precision: 51.7% — when it predicted a customer would book, it was right about half the time.
- Recall: 12.6% — it found about 13 out of every 100 actual bookings. This is the model’s main weakness.
- F1: 20.2% — a combined score for precision and recall; it’s low because the model misses many bookings.
- ROC-AUC: 77.7% — the model can separate likely bookers from non-bookers fairly well overall. This does not mean it is 77.7% accurate.
Your cross-validation results are close to your earlier test results, which is a useful comparison.

Next, create a chart showing which features were most influential, then summarise the results and the model’s weakness on your slide.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Train the full pipeline on the training data
cv_model.fit(X_train, y_train)

# Get the names and importance scores of all encoded features
feature_names = cv_model.named_steps["preprocessor"].get_feature_names_out()
importance_scores = cv_model.named_steps["classifier"].feature_importances_

# Put the results into a table
importance_table = pd.DataFrame({
    "encoded_feature": feature_names,
    "importance": importance_scores
})

# Match each encoded feature back to its original column
original_columns = sorted(X.columns, key=len, reverse=True)

def find_original_column(encoded_name):
    # Remove the transformer prefix, such as "text__"
    name = encoded_name.split("__", 1)[-1]

    # Find the original column that this feature came from
    for column in original_columns:
        if name == column or name.startswith(column + "_"):
            return column

    return name

importance_table["variable"] = importance_table["encoded_feature"].apply(
    find_original_column
)

# Add up the category scores, so the chart shows the original columns
variable_importance = (
    importance_table.groupby("variable")["importance"]
    .sum()
    .sort_values()
)

# Draw the chart
variable_importance.plot(kind="barh", figsize=(9, 6))
plt.title("Random Forest: Feature Importance")
plt.xlabel("Combined importance")
plt.ylabel("Original data column")
plt.tight_layout()
plt.show()

Your chart worked. It shows which columns the Random Forest relied on most:
- Route had the highest importance.
- Booking origin and purchase lead were next.
- Length of stay and flight hour also mattered.
- Trip type and sales channel had relatively low importance in this model.
These scores show how much the model used each column; they do not show that a column caused a customer to book. Route has many different categories, so its high score should be treated cautiously.
You now have the main results for your slide: the model’s performance scores and the feature importance chart. A key finding to mention is that it identifies some useful patterns, but its 12.6% recall means it misses most customers who book.